## Topic Modeling


# 1. Load Libraries and file



In [2]:
#install if needed
!pip install nltk gensim pyLDAvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 38.2 MB/s  0:00:000.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 35.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pyLDAvis]━━ 5/6 [pyLDAvis]nsim]


In [2]:
#basic libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
#library for pre-processing
import re
import nltk
import contractions
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [4]:
# for the topic modeling
import gensim
import gensim.corpora as corpora 
from gensim.models import LdaModel
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

In [5]:
#download necessary files for text analysis 
nltk.download('wordnet') # for lemmatization
nltk.download('punkt') # for tokenization
nltk.download('stopwords') # for stopwords
nltk.download('punkt_tab')



[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/andyzheng/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /Users/andyzheng/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/andyzheng/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/andyzheng/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
# Load file
df = pd.read_csv('data/Hotel_Reviews.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    10000 non-null  str    
 1   dateAdded             10000 non-null  str    
 2   dateUpdated           10000 non-null  str    
 3   address               10000 non-null  str    
 4   categories            10000 non-null  str    
 5   primaryCategories     10000 non-null  str    
 6   city                  10000 non-null  str    
 7   country               10000 non-null  str    
 8   keys                  10000 non-null  str    
 9   latitude              10000 non-null  float64
 10  longitude             10000 non-null  float64
 11  name                  10000 non-null  str    
 12  postalCode            10000 non-null  str    
 13  province              10000 non-null  str    
 14  reviews.date          10000 non-null  str    
 15  reviews.dateSeen      10000 non

## EDA Check missing data


In [7]:
df.isnull().sum()

id                         0
dateAdded                  0
dateUpdated                0
address                    0
categories                 0
primaryCategories          0
city                       0
country                    0
keys                       0
latitude                   0
longitude                  0
name                       0
postalCode                 0
province                   0
reviews.date               0
reviews.dateSeen           0
reviews.rating             0
reviews.sourceURLs         0
reviews.text               1
reviews.title              1
reviews.userCity        5836
reviews.userProvince    1507
reviews.username           0
sourceURLs                 0
websites                   0
dtype: int64

In [8]:
# Fill missing text with Empty space
df['reviews.text'] = df['reviews.text'].fillna('').astype(str)
df['reviews.text'].info()

<class 'pandas.Series'>
RangeIndex: 10000 entries, 0 to 9999
Series name: reviews.text
Non-Null Count  Dtype
--------------  -----
10000 non-null  str  
dtypes: str(1)
memory usage: 78.3 KB


In [9]:
#change datatypes
df['categories'] = df['categories'].astype('category')

df['primaryCategories'] = df['primaryCategories'].astype('category')

df['city'] = df['city'].astype('category')

df['dateAdded'] = pd.to_datetime(df['dateAdded'])
df['dateUpdated'] = pd.to_datetime(df['dateUpdated'])

#df['reviews.date'] = pd.to_datetime(df['reviews.date'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   id                    10000 non-null  str                
 1   dateAdded             10000 non-null  datetime64[us, UTC]
 2   dateUpdated           10000 non-null  datetime64[us, UTC]
 3   address               10000 non-null  str                
 4   categories            10000 non-null  category           
 5   primaryCategories     10000 non-null  category           
 6   city                  10000 non-null  category           
 7   country               10000 non-null  str                
 8   keys                  10000 non-null  str                
 9   latitude              10000 non-null  float64            
 10  longitude             10000 non-null  float64            
 11  name                  10000 non-null  str                
 12  postalCode      

## Pre-Processing


In [10]:
# initiate Lemmaatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
#exclude negation words
negation = {'no', 'not', 'nor', 'never'}
stop_words = stop_words - negation
#add new stopwords you like
stop_words. update(['would', 'us', 'one'])



In [11]:
#text transformations
df['cleaned_reviews'] = df['reviews.text'].str.lower() #make it lowercase 
#expand contractions (e.g., don't  --> do not)
df['cleaned_reviews'] = df['cleaned_reviews'].apply(contractions.fix)
df['cleaned_reviews'] = df['cleaned_reviews'].str.replace(r'[^a-z\s]', '', regex = True) # remove special characters
df['cleaned_reviews'] = df['cleaned_reviews'].str.split().str.join(" ") # remove empty space 

In [12]:
#Tokenization
df['tokenized_reviews'] = df['cleaned_reviews'].apply(word_tokenize)

In [13]:
#remove stopwords
df['tokenized_reviews'] = df['tokenized_reviews'].apply(lambda words: [word for word in words if word not in stop_words]) 

In [14]:
# to fully execute emmatization you need to assign a proper word tag(part of speed or POS)
# This allows the lemmatizer to know whether a word is a verb, adjective, or noun
from nltk import pos_tag
from nltk.corpus import wordnet
nltk.download('averaged_perceptron_tagger_eng')
#function to map POS tag to word
def get_wordnet_pos(treebank_tag): 
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN 
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/andyzheng/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [15]:

#lemmatization
df['tokenized_reviews'] = df['tokenized_reviews'].apply(
    lambda words : [lemmatizer.lemmatize(word, get_wordnet_pos(pos)) 
                    for word, pos in pos_tag(words)])

In [16]:
df[['reviews.text', 'cleaned_reviews', 'tokenized_reviews']]

,reviews.text,cleaned_reviews,tokenized_reviews
0,"After several hours on the road, decided to sh...",after several hours on the road decided to shu...,"[several, hour, road, decide, shut, columbus, ..."
1,This was a very beautiful hotel. The rooms wer...,this was a very beautiful hotel the rooms were...,"[beautiful, hotel, room, small, clean, quaint,..."
2,"First time in New Orleans, planned to stay her...",first time in new orleans planned to stay here...,"[first, time, new, orleans, plan, stay, night,..."
3,This is another typical motel run down by the ...,this is another typical motel run down by the ...,"[another, typical, motel, run, ownership, pers..."
4,"Nice place to stay, neat and staff are very ho...",nice place to stay neat and staff are very hom...,"[nice, place, stay, neat, staff, homely, corte..."
...,...,...,...
9995,Good: Excellent location and good price for mo...,good excellent location and good price for mon...,"[good, excellent, location, good, price, money..."
9996,Good: The staff was very friendly and helpful ...,good the staff was very friendly and helpful w...,"[good, staff, friendly, helpful, ask, directio..."
9997,Bad: No microwave. Good: The shower was amazing,bad no microwave good the shower was amazing,"[bad, no, microwave, good, shower, amazing]"
9998,Bad: Needs Security by the pool had a very bad...,bad needs security by the pool had a very bad ...,"[bad, need, security, pool, bad, experience, t..."


In [24]:
#optional step to remove short tokens
df['tokenized_reviews'] = df['tokenized_reviews'].apply(
    lambda words: [w for w in words if len(w) > 2]
)


In [25]:
# create a dictionary 
id2word = corpora.Dictionary(df['tokenized_reviews'])

In [26]:
#create corpus
corpus = [id2word.doc2bow(text) for text in df['tokenized_reviews']]

## LDA Tuning

In [ ]:
topic_range = range (5, 9)
coherence_scores =[]
for k in topic_range:
    lda_temp = LdaModel (corpus = corpus,
                      id2word = id2word, 
                      num_topics = k, 
                      passes=10, 
                      random_state=11)
    
    coherence_model = CoherenceModel(
        model = lda_temp, 
        texts = df['tokenized_reviews'], 
        dictionary=id2word, 
        coherence = 'c_v'
    )
        
    coherence_scores.append(coherence_model.get_coherence())    

#show coherence scores
for k, score in zip(topic_range, coherence_scores): 
    print (f"Number of Topics = {k}, Coherence Score = {score:.4f}")

NameError: name 'score' is not defined

In [ ]:
#plot the coherence scores 
plt.plot(list(topic_range), coherence_scores, market = 0)

## LDA Model


In [ ]:
# define the number of topics (will be adjusted later)
num_topics = 5
# train LDA model
lda_model = LdaModel (corpus = corpus,
                      id2word = id2word, 
                      num_topics = num_topics, 
                      passes=10, random_state=11)
#display the top words in each topic
for idx, topic in lda_model.print_topics(num_words = 10):
    print(f'topic {idx}: {topic}')

topic 0: 0.024*"hotel" + 0.023*"stay" + 0.014*"staff" + 0.012*"service" + 0.010*"make" + 0.009*"guest" + 0.009*"great" + 0.008*"time" + 0.008*"thank" + 0.008*"take"
topic 1: 0.039*"good" + 0.035*"hotel" + 0.026*"location" + 0.018*"great" + 0.017*"staff" + 0.016*"walk" + 0.015*"restaurant" + 0.015*"breakfast" + 0.014*"close" + 0.014*"park"
topic 2: 0.043*"stay" + 0.028*"great" + 0.025*"hotel" + 0.020*"staff" + 0.015*"time" + 0.012*"place" + 0.010*"service" + 0.009*"best" + 0.009*"visit" + 0.009*"always"
topic 3: 0.041*"room" + 0.024*"hotel" + 0.022*"clean" + 0.020*"stay" + 0.019*"breakfast" + 0.019*"nice" + 0.017*"staff" + 0.016*"good" + 0.016*"not" + 0.014*"bed"
topic 4: 0.047*"not" + 0.039*"room" + 0.016*"hotel" + 0.013*"get" + 0.011*"desk" + 0.011*"stay" + 0.010*"check" + 0.010*"front" + 0.009*"bad" + 0.009*"night"


In [ ]:
#LDA tuning
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis


In [ ]:
#enable for notebook display
pyLDAvis.enable_notebook()

In [ ]:
def visualize_lda(lda_model)